# Heritability: from family data and from summary statistics

**Purpose.** Heritability is the fraction of the variation in a trait that is
explained by genetics. This notebook estimates it twice, by two very different
routes, and the comparison is the point of the exercise.

1. **GCTA on genotypes.** With a genetic relationship matrix built from the SNPs
   themselves, a mixed model splits the phenotypic variance into a genetic and a
   residual part. This needs individual-level data.
2. **LD score regression on summary statistics.** With nothing but a published
   GWAS, the relationship between a SNP's LD score and its test statistic
   separates **real polygenic signal** from **confounding** such as population
   structure or cryptic relatedness. This is why an inflated QQ plot is not by
   itself evidence of a problem — LD score regression tells you which of the two
   you are looking at.

Along the way you will also run an association test that accounts for
relatedness, and see what the inflation factor $\lambda_{GC}$ does and does not
tell you.

**The data — two datasets, one per half.**

- **Part 1, family data.** A simulated human cohort of **related individuals**
  with a quantitative phenotype, as **called genotypes** in PLINK binary format
  (`quantfamdata.bed/.bim/.fam`). The relatedness is what makes it a family
  dataset and what makes a naive association test fail.
- **Part 2, summary statistics.** Published GWAS summary statistics from
  **Biobank Japan** for two cholesterol measurements, **HDL** and **LDL**
  (`BBJ_HDLC`, `BBJ_LDLC`) — an **East Asian** cohort, which is why the LD
  reference panel used below is the East Asian one (`eas_ldscores`). No genotypes
  are involved in this half.

## Setup

All the paths used by this exercise are set in the cell below.

In [ ]:
#############################################################
# ALL PATHS ARE SET HERE
# If the data or the software moves, this is the ONLY cell you
# need to change. No cell below this one uses a full path.
#############################################################
DATA=/course/data/current_data/heritability_ldscore
SCRIPTS=/course/data/current_data/scripts
GENETIC_MAP=/course/data/current_data/geneticMap/hg19

# LD score regression. NOTE: the conda environment this was written for is not
# on this server. The commands are left in place and the *munged* files they
# produce are supplied with the data, so the rest of the exercise still works.
LDSC_DIR=/home/jonas/anders_neededConda/ldsc
LDSC_ENV="conda run -n ldsc"
LDSC_MUNGE="${LDSC_ENV} ${LDSC_DIR}/munge_sumstats.py"
LDSC="${LDSC_ENV} ${LDSC_DIR}/ldsc.py"

WORK_DIR=$HOME/heritability_ldscore_human
mkdir -p $WORK_DIR
cd $WORK_DIR

# R and Python cannot read bash variables, so write the paths to a file they can read
cat > $WORK_DIR/env.sh <<EOF
export DATA=$DATA
export SCRIPTS=$SCRIPTS
export WORK=$WORK_DIR
EOF

ls -l $DATA/

In [ ]:
# R cannot source env.sh, so read the paths out of it rather than repeating them
env <- readLines(path.expand("~/heritability_ldscore_human/env.sh"))
getvar <- function(k) sub(paste0('^export ', k, '='), '', grep(paste0('^export ', k, '='), env, value = TRUE)[1])
DATA <- getvar("DATA"); SCRIPTS <- getvar("SCRIPTS"); WORK <- getvar("WORK")
setwd(WORK)
source(file.path(SCRIPTS, "online.R"))

In [ ]:
# Python reads the same file
import os
env = dict(l.strip().removeprefix("export ").split("=", 1)
           for l in open(os.path.expanduser("~/heritability_ldscore_human/env.sh")) if "=" in l)
DATA, SCRIPTS, WORK = env["DATA"], env["SCRIPTS"], env["WORK"]
os.chdir(WORK)

# Part 1: heritability from family data

Unpack the genotypes.

In [ ]:
source ~/heritability_ldscore_human/env.sh
cd $WORK

cp -f $DATA/quantfam.zip .
unzip -o quantfam.zip

echo ----- files in folder -----
ls

You should now have three PLINK binary files in your directory:
`quantfamdata.bed`, `quantfamdata.bim` and `quantfamdata.fam`.

The phenotype is in the sixth column of the `.fam` file. GCTA wants it in a
separate file, so extract it.

In [ ]:
fam <- read.table("./quantfamdata.fam", header = FALSE)
pheno <- data.frame(fam[,1:2], fam[,6])
write.table(pheno, file = "./myphenos.txt", quote = FALSE, row.names = FALSE, col.names = FALSE)

cat("individuals:", nrow(fam), "\n")
hist(pheno[,3], main = "the quantitative phenotype", xlab = "phenotype")

- How many individuals are there?
- Does the phenotype look normally distributed? A mixed model assumes the residuals are.

## Step 2: association testing that accounts for relatedness

GCTA's `--mlma` fits a mixed linear model, so the relatedness between individuals is modelled rather than ignored.

In [ ]:
source ~/heritability_ldscore_human/env.sh
cd $WORK

# 1.5 min to run
gcta64 --mlma --bfile quantfamdata --pheno myphenos.txt --out GCTAresults

### Load and visualise the results in R

In [ ]:
res <- read.table("./GCTAresults.mlma", header = TRUE)
head(res)

- One row per SNP. Which columns are the effect size, its standard error and the p-value?

In [ ]:
manPlot(res$p, chr = res$Chr)

In [ ]:
qqp(res$p)

chi <- qchisq(res$p, 1, lower.tail = FALSE)
lambda <- median(chi) / qchisq(0.5, 1)
cat("\nInflation factor (lambda):", lambda, "\n")

- Is $\lambda$ close to 1?
- The mixed model was supposed to absorb the relatedness. Does the QQ plot suggest it did?

### Estimate SNP Heritability with GCTA


To use GCTA to estimate the heritability accounted for by all autosomal
genome-wide SNPs, you need to first estimate the GRM, and then use the
GRM to estimate the (SNP) heritability. This can be achieved using the
following commands:

In [ ]:
source ~/heritability_ldscore_human/env.sh
cd $WORK

gcta64 --bfile quantfamdata --autosome --make-grm-bin --out GCTAgrm

The GRM holds the realised genetic relatedness between every pair of individuals. Use it to identify the individuals who are **not** related.

In [ ]:
source ~/heritability_ldscore_human/env.sh
cd $WORK

gcta64 --grm GCTAgrm --grm-singleton 0.05 --out unrelated

**Questions:**
1. How many individuals are unrelated in this sample at a GRM cut-off of 0.05? And at 0.025?
2. Try both cut-offs by changing the number in the cell above. Which cut-off would you choose, and what do you lose by choosing the stricter one?

Now estimate the SNP heritability with REML.

In [ ]:
source ~/heritability_ldscore_human/env.sh
cd $WORK

gcta64 --reml --grm-bin GCTAgrm --pheno myphenos.txt --out GCTAherit

**The screen output estimates the SNP heritability V(G)/Vp. What is it?**

- V(G) is the variance explained by the genotyped SNPs, Vp the total phenotypic variance.
- The estimate comes with a standard error. Is the estimate far enough from 0 to be convincing?
- This is **SNP heritability**, not the heritability a twin study would report. Why would you expect the twin estimate to be larger?

# Part 2: LD score regression

The rest of the notebook is based on https://cloufield.github.io/GWASTutorial/08_LDSC/.

LD score regression estimates, from summary statistics alone:

- **inflation** — and, crucially, how much of it is confounding rather than polygenicity
- **heritability** on the observed scale
- **genetic correlation** between two traits

The idea: under true polygenic inheritance, a SNP that tags many other SNPs (a
high **LD score**) should have a larger test statistic, because it picks up the
effects of everything it is correlated with. Confounding such as population
structure, by contrast, lifts *every* SNP equally regardless of its LD score. So
regressing the test statistic on the LD score separates the two: the **slope** is
polygenic signal and gives the heritability, and the **intercept** is confounding.

### Exploring the summary statistics

The `.txt.gz` files hold GWAS summary statistics for HDL and LDL cholesterol from Biobank Japan. LDL is a risk biomarker for cardiovascular disease.

In [ ]:
source ~/heritability_ldscore_human/env.sh
cd $WORK

cp -f $DATA/BBJ*.gz .
cp -f $DATA/w_hm3.snplist .
ln -sf $DATA/eas_ldscores .

echo "number of lines for HDLC:"
zcat BBJ_HDLC.txt.gz | wc -l

echo "number of lines for LDLC:"
zcat BBJ_LDLC.txt.gz | wc -l

Let's look at the first 10 lines of the summary statistics for HDL.

In [ ]:
source ~/heritability_ldscore_human/env.sh
cd $WORK
zcat BBJ_HDLC.txt.gz | head -n 10 | column -t

- Which columns would LD score regression need, and which are extra?

### Visualise the summary statistics

Before running LD score regression, look at the results in R.

In [ ]:
# load package for fast reading of data
suppressMessages(require(data.table))
suppressMessages(require(R.utils))

data <- fread("./BBJ_HDLC.txt.gz", sep = "\t", header = TRUE, stringsAsFactors = FALSE, data.table = FALSE)
head(data)

Let's make a Manhattan plot.

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 6)
manPlot(data$P, chr = as.integer(data$CHR), cap = 1e-30, main = "HDLC")

- How many signals are there? Compare this with the Manhattan plot from Part 1 — what does the difference in sample size do to the picture?

Let's zoom in on the strongest signal.

In [ ]:
w <- which.min(data$P)
pos <- data$POS[w]
chr <- data$CHR[w]

win <- 5e4 # <- change this to zoom in or out
region <- subset(data, CHR == chr & POS > pos-win & POS < pos+win)

plot(region$POS, -log10(region$P), xlab = paste("position on chr", chr),
     ylab = "-log10(p)", main = "strongest HDLC signal", pch = 16, col = "darkblue")

- Based on the plot, what is the candidate gene? (Look up the position.)
- Can we be sure it is the causal gene?
- The lowest p-value is extremely small. What does that say about the effect size, and about the sample size?

In [ ]:
manPlot(data$P, chr = as.integer(data$CHR), cap = 1e-30, main = "HDLC")

- How many chromosomes have at least one genome-wide significant SNP?

In [ ]:
qqPlot(data$P, cap = 1e-300, main = "HDLC")

# calculate the inflation factor
X2 <- qchisq(data$P, lower = FALSE, df = 1)
lambda <- median(X2) / qchisq(0.5, 1)
cat("lambda GC =", lambda, "\n")

- How does the QQ plot look? Anything to be worried about?
- Try changing the `cap` option to see the shape of the curve better.
- $\lambda$ is well above 1. Two things could cause that: real polygenic signal, or confounding. The QQ plot cannot tell them apart — that is exactly what LD score regression is for.

In [ ]:
dataLDL <- fread("./BBJ_LDLC.txt.gz", sep = "\t", header = TRUE, stringsAsFactors = FALSE, data.table = FALSE)
qqPlot(dataLDL$P, cap = 1e-300, main = "LDLC")

X2 <- qchisq(dataLDL$P, lower = FALSE, df = 1)
cat("lambda GC =", median(X2) / qchisq(0.5, 1), "\n")

- Are the LDL p-values inflated too?
- Which of the two traits looks more inflated, and would you conclude anything from that on its own?

### Step 1: munge the summary statistics

Before the regression, the raw summary statistics have to be reformatted and
restricted to a well-behaved set of SNPs — the HapMap3 list in `w_hm3.snplist`,
which is what the LD scores were computed on.

> **⚠️ The conda environment for LDSC is not installed on this server**, so the
> two `munge_sumstats.py` cells below will fail. Their output
> (`BBJ_HDLC.sumstats.gz` and `BBJ_LDLC.sumstats.gz`) is supplied with the data,
> so everything after them still runs. Read the commands, then move on.

In [ ]:
source ~/heritability_ldscore_human/env.sh
cd $WORK

# takes around 1 min
${LDSC_MUNGE} \
    --sumstats ./BBJ_HDLC.txt.gz \
    --merge-alleles ./w_hm3.snplist \
    --a1 ALT \
    --a2 REF \
    --chunksize 500000 \
    --out BBJ_HDLC

- How many SNPs merged with the pre-calculated LD measurements, and how many were dropped?
- The inflation factor reported here differs slightly from the one you calculated above. Why? (Think about which SNPs survived the merge.)

In [ ]:
source ~/heritability_ldscore_human/env.sh
cd $WORK

${LDSC_MUNGE} \
    --sumstats ./BBJ_LDLC.txt.gz \
    --merge-alleles ./w_hm3.snplist \
    --a1 ALT \
    --a2 REF \
    --chunksize 500000 \
    --out BBJ_LDLC

After munging you have two formatted files that can be matched against the LD reference data:

In [ ]:
source ~/heritability_ldscore_human/env.sh
cd $WORK

# the munged files are supplied with the data, so copy them in
cp -f $DATA/BBJ_HDLC.sumstats.gz $DATA/BBJ_LDLC.sumstats.gz .

zcat BBJ_HDLC.sumstats.gz | head -n 2
zcat BBJ_LDLC.sumstats.gz | head -n 2

echo "---- number of lines ----"
zcat BBJ_HDLC.sumstats.gz | wc -l
zcat BBJ_LDLC.sumstats.gz | wc -l

- The munged file has far fewer columns than the raw one. Which ones survived, and why are those the only ones LD score regression needs?

### LD score regression

Univariate LD score regression estimates heritability **and** the confounding
factors — population stratification, cryptic relatedness — at the same time.

The LD scores are the East Asian ones, matching the ancestry of Biobank Japan.

In [ ]:
source ~/heritability_ldscore_human/env.sh
cd $WORK

${LDSC} \
  --h2 BBJ_HDLC.sumstats.gz \
  --ref-ld-chr eas_ldscores/ \
  --w-ld-chr eas_ldscores/ \
  --out BBJ_HDLC

- Why does the LD reference panel have to match the ancestry of the GWAS? What would go wrong with a European panel here?

In [ ]:
source ~/heritability_ldscore_human/env.sh
cd $WORK

${LDSC} \
  --h2 BBJ_LDLC.sumstats.gz \
  --ref-ld-chr eas_ldscores/ \
  --w-ld-chr eas_ldscores/ \
  --out BBJ_LDLC

Let's check the results for HDL:

In [ ]:
source ~/heritability_ldscore_human/env.sh
cd $WORK
cat BBJ_HDLC.log

The log reports, for HDL:

- observed-scale $h^2$ = 0.1583
- $\lambda_{GC}$ = 1.1523
- intercept = 1.0563
- ratio = 0.1981

**Questions:**
- The intercept is 1.0563, not 1. What does the excess over 1 represent?
- The ratio is (intercept − 1) / (mean $\chi^2$ − 1) — the fraction of the inflation that is **not** polygenic signal. At 0.198, how much of the inflation seen in the QQ plot is confounding?
- $\lambda_{GC}$ is 1.15 and the intercept is 1.06. Explain in one sentence why those two numbers differ, and why the intercept is the more useful of the two.
- Now look at the LDL log. Which trait is more heritable, and which has more confounding?

### Run the cell below to take the quiz

In [ ]:
from jupyterquiz import display_quiz

display_quiz("https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/gwas/quiz/heritability_ldscore.json")